###Spark Aggregate Functions

#####Funciones simples de agregación

In [0]:
%run '../includes/configuration'

In [0]:
movies_df = spark.read.parquet(f"{silver_folder_path}/movies")

In [0]:
from pyspark.sql import functions as F, window as W

In [0]:
movies_df.select(F.count("*")).show()

+--------+
|count(1)|
+--------+
|    4725|
+--------+



In [0]:
movies_df.select(F.count("year_release_date")).show()

In [0]:
display(movies_df.select(F.countDistinct("year_release_date")))

In [0]:
display(movies_df.select(F.sum("budget")))

In [0]:
movies_df.filter("year_release_date = 2016") \
            .select(F.sum("budget"), F.count("movie_id")) \
            .withColumnRenamed("sum(budget)","total_budget") \
            .withColumnRenamed("count(movie_id)","count_movies") \
            .display()

total_budget,count_movies
4.29015E9,94


In [0]:
movie_group_by_df = movies_df \
.groupBy("year_release_date")\
    .agg(
        F.sum("budget").alias("total_budget"),
        F.avg("budget").alias("avg_budget"),
        F.max("budget").alias("max_budget"),
        F.min("budget").alias("min_budget"),
        F.count("movie_id").alias("count_movies"),
    )

In [0]:
movie_group_by_df.display()

year_release_date,total_budget,avg_budget,max_budget,min_budget,count_movies
null,7.889118615E9,2.6385012090301003E7,95000000,100000,299
1916,385907.0,385907.0,385907,385907,1
1925,245000.0,245000.0,245000,245000,1
1927,9.262E7,9.262E7,92620000,92620000,1
1929,2379000.0,1189500.0,379000,2000000,2
1930,3950000.0,3950000.0,3950000,3950000,1
1932,2000000.0,2000000.0,2000000,2000000,1
1933,439000.0,439000.0,439000,439000,1
1934,325000.0,325000.0,325000,325000,1
1935,609000.0,609000.0,609000,609000,1


In [0]:
movie_rank = W.Window.partitionBy("year_release_date").orderBy(F.desc("budget"))
movie_dense_rank = W.Window.partitionBy("year_release_date").orderBy(F.desc("budget"))

movies_df.select("title","budget","year_release_date") \
    .filter("year_release_date is not null") \
    .withColumn("rank",F.rank().over(movie_rank)) \
    .withColumn("dense_rank",F.dense_rank().over(movie_dense_rank)) \
    .withColumn("perfect_rank",F.row_number().over(movie_rank)) \
    .display()



title,budget,year_release_date,rank,dense_rank,perfect_rank
Intolerance,385907,1916,1,1,1
The Big Parade,245000,1925,1,1,1
Metropolis,92620000,1927,1,1,1
The Broadway Melody,379000,1929,1,1,1
Die Büchse der Pandora,2000000,1929,2,2,2
Hell's Angels,3950000,1930,1,1,1
A Farewell to Arms,2000000,1932,1,1,1
42nd Street,439000,1933,1,1,1
It Happened One Night,325000,1934,1,1,1
Top Hat,609000,1935,1,1,1
